# Organizational Catalog Schema Test

Open this notebook in **ArcGIS Pro** and use its managed Python kernel. It creates only `_schema_notebook_probe.gdb` beside `Vertex_Assets.gdb`; it does not modify the production source or create the final organizational catalog.

Run the cells in order. Notebook testing begins with the environment check in Cell 2. The key failing boundary is isolated in Cell 5.

In [1]:
from pathlib import Path
import os
import shutil
import sys

import arcpy

REPOSITORY = Path(r"c:\dev\sandbox\demo-data-generator")
SOURCE_GDB = Path(r"c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Vertex_Assets.gdb")
PROBE_GDB = SOURCE_GDB.parent / "_schema_notebook_probe.gdb"

sys.path.insert(0, str(REPOSITORY / "src"))

from indoors_demo_generator.preflight import validate_indoors_source
from indoors_demo_generator.schema import create_asset_management_schema

print("ArcGIS Pro:", arcpy.GetInstallInfo()["Version"])
print("Repository:", REPOSITORY)
print("Source:", SOURCE_GDB)
print("Disposable probe:", PROBE_GDB)

ArcGIS Pro: 3.7.2
Repository: c:\dev\sandbox\demo-data-generator
Source: c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\Vertex_Assets.gdb
Disposable probe: c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb


In [2]:
if arcpy.Exists(str(PROBE_GDB)):
    arcpy.management.Delete(str(PROBE_GDB))
elif PROBE_GDB.exists():
    shutil.rmtree(PROBE_GDB)

arcpy.management.CreateFileGDB(str(PROBE_GDB.parent), PROBE_GDB.name)
source_indoors = SOURCE_GDB / "Indoors"
target_indoors = PROBE_GDB / "Indoors"
spatial_reference = arcpy.Describe(str(source_indoors / "Units")).spatialReference
arcpy.management.CreateFeatureDataset(str(PROBE_GDB), "Indoors", spatial_reference)

for name in ("Sites", "Facilities", "Levels", "Units"):
    target = str(target_indoors / name)
    arcpy.management.CopyFeatures(str(source_indoors / name), target)
    arcpy.management.AddGlobalIDs(target)
    global_id_type = next(
        field.type for field in arcpy.ListFields(target) if field.name.upper() == "GLOBALID"
    )
    print(name, "rows=", arcpy.management.GetCount(target)[0], "GlobalID=", global_id_type)

source = validate_indoors_source(arcpy, str(PROBE_GDB))
print("Minimal Indoors validation passed")

Sites rows= 1 GlobalID= GlobalID
Facilities rows= 1 GlobalID= GlobalID
Levels rows= 5 GlobalID= GlobalID
Units rows= 286 GlobalID= GlobalID
Minimal Indoors validation passed


## Schema Creation Boundary

The external ArcGIS Python process completes the minimal hierarchy, then terminates while creating the generator-owned schema. Cell 5 repeats only that boundary.

If Cell 5 reports an error or restarts the kernel, retain the complete geoprocessing message and identify the last dataset, domain, or relationship visible in `_schema_notebook_probe.gdb`.

In [3]:
try:
    schema = create_asset_management_schema(arcpy, str(PROBE_GDB), source)
except Exception:
    print(arcpy.GetMessages())
    raise
else:
    print("Schema creation passed")
    for name, path in vars(schema).items():
        print(name, "=", path)

Schema creation passed
feature_dataset = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetManagement
indoor_asset_locations = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetManagement\IndoorAssetLocation
exterior_asset_points = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetManagement\ExteriorAssetPoints
asset_lifecycle = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetLifecycle
inspections = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetInspection
repairs = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetRepair
work_orders = c:\Workspace\Sandbox\data\ILM_asset_gen\Medical Indoors_3141f0\GDBs\_schema_notebook_probe.gdb\AssetWorkOrder
external_references = c:\Workspace\Sandbox\

In [4]:
print("ALERT: integrated schema validation begins here")

inventory = []
for root, _, names in arcpy.da.Walk(
    str(PROBE_GDB), datatype=["FeatureClass", "Table", "RelationshipClass"]
):
    for name in names:
        path = os.path.join(root, name)
        inventory.append(os.path.relpath(path, str(PROBE_GDB)))

for item in sorted(inventory):
    print(item)

domains = sorted(domain.name for domain in arcpy.da.ListDomains(str(PROBE_GDB)))
print("Domains:", domains)

required = {
    "AssetManagement\\IndoorAssetLocation",
    "AssetManagement\\ExteriorAssetPoints",
    "AssetLifecycle",
    "AssetInspection",
    "AssetRepair",
    "AssetWorkOrder",
}
assert required.issubset(set(inventory)), required - set(inventory)
assert "AssetCondition" in domains
assert "AssetClass" in domains
print("Notebook schema test completed successfully")

ALERT: integrated schema validation begins here
AssetExternalReference
AssetFieldWorkLink
AssetInspection
AssetLifecycle
AssetLifecycle_ExternalReferences
AssetLifecycle_FieldWorkLinks
AssetLifecycle_Inspections
AssetLifecycle_MediaReferences
AssetLifecycle_Repairs
AssetLifecycle_WorkOrders
AssetManagement\ExteriorAssetPoints
AssetManagement\IndoorAssetLocation
AssetMediaReference
AssetRepair
AssetWorkOrder
Facilities_ExteriorAssetPoints
IndoorLocations_AssetLifecycle
Indoors\Facilities
Indoors\Levels
Indoors\Sites
Indoors\Units
Units_IndoorAssetLocations
Domains: ['AssetClass', 'AssetCondition', 'AssetMediaType', 'AssetPlacementType', 'CriticalityRating', 'ExternalSystem', 'FieldWorkStatus', 'FieldWorkWorkflow', 'InspectionCondition', 'IntegrationSyncStatus', 'WorkOrderStatus', 'YesNo']
Notebook schema test completed successfully


In [ ]:
import runpy

BUILDER = REPOSITORY / "scripts" / "build_organizational_asset_catalog.py"
OUTPUT_GDB = SOURCE_GDB.parent / "Organizational_Asset_Catalog.gdb"
MANIFEST = SOURCE_GDB.parent / "Organizational_Asset_Catalog_manifest.json"
FORCE_REBUILD = False

output_exists = arcpy.Exists(str(OUTPUT_GDB)) or OUTPUT_GDB.exists()
if output_exists and not FORCE_REBUILD:
    if not MANIFEST.exists():
        raise FileExistsError(
            f"Catalog exists without its manifest: {OUTPUT_GDB}. "
            "Inspect it before setting FORCE_REBUILD = True."
        )
    print("Validated production catalog already exists; rebuild skipped:", OUTPUT_GDB)
else:
    arcpy.ClearWorkspaceCache_management()
    if output_exists:
        if not FORCE_REBUILD:
            raise RuntimeError("Rebuild requires FORCE_REBUILD = True.")
        if arcpy.Exists(str(OUTPUT_GDB)):
            arcpy.management.Delete(str(OUTPUT_GDB))
        elif OUTPUT_GDB.exists():
            shutil.rmtree(OUTPUT_GDB)
    print("Source remains read-only:", SOURCE_GDB)
    print("Production output:", OUTPUT_GDB)
    runpy.run_path(str(BUILDER), run_name="__main__")

In [ ]:
import json

MANIFEST = SOURCE_GDB.parent / "Organizational_Asset_Catalog_manifest.json"
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

print("Validation summary")
print(json.dumps(manifest["validation"], indent=2))
print("Related table counts")
print(json.dumps(manifest["related_table_counts"], indent=2))

lifecycle = str(OUTPUT_GDB / "AssetLifecycle")
sample_fields = [
    "Asset_ID", "Name", "AssetClass", "Facility_Name", "City", "Department",
    "Condition", "LifecycleStage", "AssetAgeYears", "RemainingLifeYears",
    "EstimatedReplacementCost",
 ]
print("Representative lifecycle records")
with arcpy.da.SearchCursor(lifecycle, sample_fields) as cursor:
    for index, row in enumerate(cursor):
        print(dict(zip(sample_fields, row)))
        if index == 4:
            break

assert arcpy.Exists(str(OUTPUT_GDB))
assert manifest["validation"]["facilities"] == 6
assert manifest["validation"]["admin_hq_rooms"] == 572
assert manifest["validation"]["orphan_references"] == 0
print("Production catalog verification passed")